# IndicPulse – MuRIL Fine-tuning Notebook
## Fine-tune `google/muril-base-cased` for 4-class Hinglish Sentiment Classification

**Runtime:** Google Colab T4 GPU (Free Tier)  
**Duration:** ~20-30 minutes for 3 epochs on 2000 samples

### Label Schema
| ID | Class | Description |
|---|---|---|
| 0 | Positive | Genuine praise, hype, appreciation |
| 1 | Sarcastic | Trolling, ironic, backhanded compliments |
| 2 | Neutral | Informational, questions, mixed feedback |
| 3 | Toxic | Abusive, profane, hate speech |

In [ ]:
# ── Step 0: Verify GPU ────────────────────────────────────
import torch
print(f'GPU Available: {torch.cuda.is_available()}')
print(f'GPU Name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
!nvidia-smi

In [ ]:
# ── Step 1: Install dependencies ─────────────────────────
!pip install -q transformers datasets accelerate scikit-learn numpy pandas

In [ ]:
# ── Step 2: Upload your labeled dataset ──────────────────
# Your CSV must have columns: text, label
# label: 0=Positive, 1=Sarcastic, 2=Neutral, 3=Toxic

from google.colab import files
import pandas as pd

print('Upload your training_labels.csv file:')
uploaded = files.upload()
fname = list(uploaded.keys())[0]

df = pd.read_csv(fname)
print(f'Loaded {len(df)} samples')
print(df['label'].value_counts())
df.head()

In [ ]:
# ── Step 3: Tokenize dataset ─────────────────────────────
import numpy as np
from transformers import AutoTokenizer
from datasets import Dataset

MODEL_NAME = 'google/muril-base-cased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

dataset = Dataset.from_pandas(df[['text', 'label']])
split = dataset.train_test_split(test_size=0.2, seed=42)

def tokenize_fn(batch):
    return tokenizer(
        batch['text'],
        truncation=True,
        max_length=256,
        padding='max_length'
    )

tokenized = split.map(tokenize_fn, batched=True)
tokenized = tokenized.rename_column('label', 'labels')
tokenized.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])

print(f'Train: {len(tokenized["train"])} | Val: {len(tokenized["test"])}')

In [ ]:
# ── Step 4: Load MuRIL model ─────────────────────────────
from transformers import AutoModelForSequenceClassification

LABEL_NAMES = ['Positive', 'Sarcastic', 'Neutral', 'Toxic']

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4,
    id2label={i: l for i, l in enumerate(LABEL_NAMES)},
    label2id={l: i for i, l in enumerate(LABEL_NAMES)},
)

model = model.cuda()
print('MuRIL loaded on GPU ✅')
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# ── Step 5: Define metrics ───────────────────────────────
from sklearn.metrics import accuracy_score, f1_score, classification_report

def compute_metrics(pred):
    logits, labels = pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    f1_macro = f1_score(labels, preds, average='macro')
    f1_weighted = f1_score(labels, preds, average='weighted')
    return {
        'accuracy': acc,
        'f1_macro': f1_macro,
        'f1_weighted': f1_weighted,
    }

In [ ]:
# ── Step 6: Configure Training ───────────────────────────
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding

OUTPUT_DIR = '/content/muril_finetuned'

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_ratio=0.1,
    weight_decay=0.01,
    learning_rate=2e-5,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    greater_is_better=True,
    fp16=True,                      # FP16 on T4 for speed
    logging_steps=50,
    report_to='none',
    save_total_limit=2,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['test'],
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)

print('Training configuration ready ✅')

In [ ]:
# ── Step 7: TRAIN! ───────────────────────────────────────
print('Starting MuRIL fine-tuning...')
print('='*50)

train_result = trainer.train()

print('\nTraining complete! ✅')
print(f'Train loss: {train_result.training_loss:.4f}')

In [ ]:
# ── Step 8: Evaluate ────────────────────────────────────
eval_results = trainer.evaluate()
print('\nValidation Results:')
for k, v in eval_results.items():
    print(f'  {k}: {v:.4f}')

# Detailed classification report
preds_output = trainer.predict(tokenized['test'])
y_pred = np.argmax(preds_output.predictions, axis=-1)
y_true = preds_output.label_ids

print('\nDetailed Classification Report:')
print(classification_report(y_true, y_pred, target_names=LABEL_NAMES))

In [ ]:
# ── Step 9: Save model ───────────────────────────────────
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'Model saved to: {OUTPUT_DIR}')

import os
for f in os.listdir(OUTPUT_DIR):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    print(f'  {f}: {size/1024/1024:.1f} MB')

In [ ]:
# ── Step 10: Download model ──────────────────────────────
import shutil
shutil.make_archive('/content/muril_finetuned', 'zip', OUTPUT_DIR)
files.download('/content/muril_finetuned.zip')
print('Download started! Extract to indicpulse/models/muril_finetuned/')

In [ ]:
# ── Step 11: Quick inference test ────────────────────────
from transformers import pipeline

classifier = pipeline(
    'text-classification',
    model=model,
    tokenizer=tokenizer,
    device=0,
    truncation=True,
    max_length=256,
)

test_comments = [
    'Bhai yeh video ekdum fire hai [fire_hype] audio bahut badhiya',
    'Haan haan bilkul expert hai yeh [clown_mocking] zyaada mat socho',
    'Audio thoda low hai kuch jagah par',
    'chutiya content creator hai yeh bekaar hai sab [angry_frustrated]',
]

print('Inference test results:')
results = classifier(test_comments)
for comment, result in zip(test_comments, results):
    print(f'  [{result["label"]} | {result["score"]:.2%}] {comment[:60]}')